# Shared-Analysis Round-Trip: publish & replicate attribution-graph steering

This notebook demonstrates end-to-end **shared analysis**: one user generates attribution-graph
sign-aware feature-steering results; a second user **replicates those results from the published
artifact with the expensive attribution-graph generation skipped entirely** — only the cheap
steering forward pass re-runs, from the stored graph.

1. **Generate locally** — the `it.intervention_from_concept` pipeline produces the reference
   results and an `AnalysisStore`.
2. **Push** — the store publishes as a Hub *dataset* repo with a machine-written
   `it_artifact.json` envelope (parquet interchange; the local format stays Arrow).
3. **Clean** — the local analysis state is deleted; only small reference snapshots remain.
4. **Replicate** — `it.hub.pull_analysis_store` fetches the artifact, the interpretune formatter
   re-attaches from the envelope, and the named analysis backend hydrates steering-capable
   graphs; the steering results are reproduced and compared against the reference.
5. **Optional coda** — a small provenance iteration re-uploads to the same repo: the artifact's
   `identity` survives, its content fingerprint tracks the change.


In [ ]:
# Parameters - These will be injected by papermill during parameterized test runs
BACKEND = "nnsight"  # circuit-tracer backend: "nnsight" or "transformerlens"
CONCEPT_PROMPT = "Is orange a color or a fruit? Answer with one word: Color or Fruit. orange ->"
CONCEPT_TARGET_TOKENS = ["Fruit", "Color"]
FEATURE_SELECTION_TOP_N = 5
FEATURE_SELECTION_MIN_LAYER = 10
FEATURE_SELECTION_SCORE_SIGN = "any"
INTERVENTION_SCALE_FACTOR = 20.0
REGISTRY_KEY = "rte_demo.gemma2.circuit_tracer"  # hub component configuration key (model + backend)
MODEL_NAME = "gemma-2-2b"
TRANSCODER_SET = "gemma"
CHAT_FORMAT_PROMPT = False

# -- The shared-analysis artifact -----------------------------------------------------------------
# Set this to YOUR OWN `<org>/<repo>` (any org you can write to). `speediedan/...` is only the
# maintainer's validation target -- nothing about the workflow is specific to it.
ARTIFACT_REPO_ID = "speediedan/ct-steering-rte-demo"
ARTIFACT_PRIVATE = True
HF_TOKEN_ENV = "IT_HF_TOKEN"  # env var holding a WRITE token for ARTIFACT_REPO_ID's org
RUN_REUPLOAD_CODA = True  # step 5: provenance-iteration re-upload (identity must survive)

In [ ]:
# @title Imports { display-mode: "form" }
import os

import torch  # noqa: F401

import interpretune.analysis  # noqa: F401  # ensure op wrappers are registered
from interpretune.analysis import analysis_store_from_batches
from interpretune.analysis.backends import FeatureSelectionSpec
from interpretune.analysis.ops.base import AnalysisBatch
from it_examples.utils.nb_ui_utils import (  # noqa: F401
    display_steering_results,
    display_target_gap,
)

## 1. Generate locally — the reference results

In [ ]:
# @title 1: Session construction { display-mode: "form" }
from pathlib import Path

from dotenv import load_dotenv

import interpretune as it
from it_examples.seeds import ensure_local_seeds
from interpretune import ITSession, ITSessionConfig

for _env_candidate in (Path.cwd() / ".env", Path.home() / "repos" / "interpretune" / ".env"):
    if _env_candidate.exists():
        load_dotenv(_env_candidate)
        break

ensure_local_seeds()  # idempotent, offline: seed publish sources -> local components cache
base_itdm_cfg, base_it_cfg, dm_cls, m_cls = it.hub.load("speediedan/rte", REGISTRY_KEY)
base_it_cfg.circuit_tracer_cfg.backend = BACKEND
if TRANSCODER_SET:
    base_it_cfg.circuit_tracer_cfg.transcoder_set = TRANSCODER_SET
if BACKEND == "nnsight":
    adapter_ctx = (it.Adapter.core, it.Adapter.nnsight, it.Adapter.circuit_tracer)
else:
    adapter_ctx = (it.Adapter.core, it.Adapter.transformer_lens, it.Adapter.circuit_tracer)
session_cfg = ITSessionConfig(
    adapter_ctx=adapter_ctx,
    datamodule_cfg=base_itdm_cfg,
    module_cfg=base_it_cfg,
    datamodule_cls=dm_cls,
    module_cls=m_cls,
)
it_session = ITSession(session_cfg)
it.it_init(**it_session)
module = it_session.module
tokenizer = module.replacement_model.tokenizer
print(f"session ready: {type(module).__name__} ({MODEL_NAME} + circuit-tracer {BACKEND} backend)")

In [ ]:
# @title 2: Run the pipeline; capture the reference; build the store { display-mode: "form" }
from interpretune.config import AnalysisCfg, init_analysis_cfgs

module.analysis_cfg = AnalysisCfg(target_op=it.intervention_from_concept, ignore_manual=True, save_tokens=False)
init_analysis_cfgs(module, [module.analysis_cfg])

fruits = ["apple", "banana", "grape", "peach"]
colors = ["red", "blue", "green", "yellow"]
if CHAT_FORMAT_PROMPT:
    from it_examples.examples.prompt_configs.prompt_configs import GemmaPromptConfig

    prompt = GemmaPromptConfig().apply_chat_template_fn(
        tokenizer, CONCEPT_PROMPT, tokenize=False, add_generation_prompt=True
    )
else:
    prompt = CONCEPT_PROMPT

ct_cfg = module.it_cfg.circuit_tracer_cfg
ct_cfg.intervention_sign_aware_scale = True
ct_cfg.intervention_max_influence_norm_scale = True
ct_cfg.intervention_value_source = "top_feature_activation_values"

selection_spec = FeatureSelectionSpec(
    layer_slice=slice(FEATURE_SELECTION_MIN_LAYER, None),
    score_source="signed_influence",
    score_sign=FEATURE_SELECTION_SCORE_SIGN,
    rank_by_abs=True,
)
pipeline_results = it.intervention_from_concept(
    module,
    AnalysisBatch(
        concept_group_a=fruits,
        concept_group_b=colors,
        concept_label="Concept: Fruit - Color",
        concept_direction_mode="paired_rejection",
        prompts=[prompt],
    ),
    None,
    0,
    top_n=FEATURE_SELECTION_TOP_N,
    intervention_scale_factor=INTERVENTION_SCALE_FACTOR,
    feature_selection=selection_spec,
)
steering = display_steering_results(pipeline_results, tokenizer, CONCEPT_TARGET_TOKENS)

# the store IS the shareable artifact: the pipeline results serialized by the op's own schema
store = analysis_store_from_batches(
    module, [pipeline_results], op=it.intervention_from_concept, analysis_backend="circuit_tracer"
)
# REFERENCE snapshots (plain python; everything else local is deleted in step 3)
reference_row = {k: v for k, v in store.dataset.with_format(None)[0].items()}
reference_columns = sorted(store.dataset.column_names)
print(f"reference captured: {len(reference_columns)} columns, 1 row")

## 2. Push the store to the Hub (parquet interchange + envelope + generated card)

In [ ]:
# @title 3: Push { display-mode: "form" }
token = os.environ[HF_TOKEN_ENV]
pushed_revision = it.hub.push_analysis_store(
    store,
    ARTIFACT_REPO_ID,
    private=ARTIFACT_PRIVATE,
    token=token,
    provenance={"generated_by": "intervention_from_concept", "model_name": MODEL_NAME, "backend": BACKEND},
)
print(f"pushed {ARTIFACT_REPO_ID} @ {pushed_revision}")

## 3. Clean the local analysis state

Everything the pipeline produced locally is deleted — what follows must come from the Hub.

In [ ]:
# @title 4: Clean { display-mode: "form" }
import gc
import shutil

local_save_dir = getattr(getattr(module.analysis_cfg, "output_store", None), "save_dir", None)
del store, pipeline_results
gc.collect()
if local_save_dir and Path(local_save_dir).exists():
    shutil.rmtree(local_save_dir)
    print(f"removed local analysis output: {local_save_dir}")
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("local analysis state cleaned -- only the reference snapshots remain in memory")

## 4. Replicate from the Hub artifact — no pipeline re-run

`pull_analysis_store` fetches envelope-first (revision-pinned), re-attaches the interpretune
formatter from the envelope, and resolves the **named** analysis backend (`circuit_tracer`) so
row access hydrates steering-capable `Graph` objects. We verify three things:

1. **Transport fidelity** — every stored column round-trips the parquet interchange intact.
2. **Hydration** — the pulled store yields a live attribution `Graph` via the backend seam.
3. **Steering replication** — the graph computation is skipped; `it.intervention_from_features`
   re-runs only the (cheap) steering forward pass from the stored graph and reproduces the
   reference steering outcome. The compute the second user avoids is exactly the expensive
   attribution analysis.

In [ ]:
# @title 5: Pull + verify { display-mode: "form" }
import numpy as np

pulled = it.hub.pull_analysis_store(ARTIFACT_REPO_ID, token=token)
envelope = it.hub.describe_analysis_store(ARTIFACT_REPO_ID)
print(f"artifact identity: {envelope['identity']['store_id']} (created {envelope['identity']['created_utc']})")
print(f"provenance: {envelope['provenance']}")

# 1) transport fidelity: every column byte-faithful through parquet
pulled_row = {k: v for k, v in pulled.dataset.with_format(None)[0].items()}
assert sorted(pulled.dataset.column_names) == reference_columns
mismatched = []
for col in reference_columns:
    ref, got = reference_row[col], pulled_row[col]
    equal = np.array_equal(np.asarray(ref, dtype=object), np.asarray(got, dtype=object))
    if not equal:
        mismatched.append(col)
assert not mismatched, f"columns changed in transport: {mismatched}"
print(f"transport fidelity: {len(reference_columns)}/{len(reference_columns)} columns identical")

# 2) hydration via the named backend seam (no generation pipeline involved)
hydrated = pulled[0]
graph = hydrated["attribution_graph"]
print(f"hydrated graph: {type(graph).__name__} ({graph.active_features.shape[0]} active features)")

# 3) steering replication: ONLY the intervention forward pass re-runs, from hydrated state
# the prompt itself travels IN the artifact (the graph's input_string) -- the second user
# needs nothing beyond the hub repo to re-run the steering forward
replicated = it.intervention_from_features(
    module,
    AnalysisBatch(**{k: v for k, v in hydrated.items()}, prompts=[graph.input_string]),
    None,
    0,
    intervention_scale_factor=INTERVENTION_SCALE_FACTOR,
)
replicated_steering = display_steering_results(replicated, tokenizer, CONCEPT_TARGET_TOKENS)
print("steering replicated: graph computation skipped; steering ran from the stored graph")

## 5. Optional coda — share an iteration

Re-uploading after a change keeps the artifact's **identity** (`store_id` never rewritten) while
its provenance — including the content fingerprint — refreshes.

In [ ]:
# @title 6: Re-upload coda { display-mode: "form" }
if RUN_REUPLOAD_CODA:
    original_identity = envelope["identity"]
    coda_revision = it.hub.push_analysis_store(
        pulled,
        ARTIFACT_REPO_ID,
        private=ARTIFACT_PRIVATE,
        token=token,
        provenance={"generated_by": "intervention_from_concept", "iteration_note": "round-trip coda re-upload"},
    )
    refreshed = it.hub.pull_analysis_store(ARTIFACT_REPO_ID, token=token)  # refresh the cached envelope
    coda_envelope = it.hub.describe_analysis_store(ARTIFACT_REPO_ID)
    assert coda_envelope["identity"] == original_identity, "identity must survive re-upload"
    print(f"re-uploaded @ {coda_revision}; identity preserved: {coda_envelope['identity']['store_id']}")
    print(f"iteration note: {coda_envelope['provenance'].get('iteration_note')}")
else:
    print("coda skipped (RUN_REUPLOAD_CODA=False)")

## Summary

- The `it.intervention_from_concept` results became a **portable artifact**: parquet interchange +
  `it_artifact.json` envelope + generated dataset card, published with `it.hub.push_analysis_store`.
- After deleting all local analysis state, `it.hub.pull_analysis_store` reproduced the exact stored
  columns, hydrated a steering-capable attribution `Graph` through the **named backend seam**, and
  `it.intervention_from_features` replicated the steering outcome — **the attribution-graph
  generation never re-ran; only the steering forward pass did, from the stored graph**.
- Re-uploading kept the artifact's identity while its content fingerprint tracked the change: the
  shared-iteration story.
